# 🌟 TarjimaAI: English-to-Uzbek Neural Machine Translation (Seq2Seq)
### **Google Colab GPU Accelerated Training & Google Drive Persistence**

This notebook trains both:
1. **Baseline Seq2Seq** (Sutskever et al. 2014 bottleneck)
2. **Bahdanau Additive Attention Seq2Seq** (Bahdanau et al. 2015)

Trained on 48,000 English-Uzbek parallel pairs with automatic saving of trained `.pth` model checkpoints and vocabulary mappings to **Google Drive**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create destination directory in your Google Drive for permanent storage
DRIVE_DIR = '/content/drive/MyDrive/eng_uzb_translator_checkpoints'
!mkdir -p "{DRIVE_DIR}"
print(f"Google Drive checkpoint destination ready: {DRIVE_DIR}")

## 2. Verify GPU Acceleration (NVIDIA CUDA)

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 3. Clone Repository from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/hamzayevtemur-lab/eng-uzb-translator.git"
REPO_NAME = "eng-uzb-translator"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL} /content/{REPO_NAME}
else:
    %cd /content/{REPO_NAME}
    !git pull origin main

%cd /content/{REPO_NAME}
!ls -lh

## 4. Install Dependencies

In [ ]:
%cd /content/{REPO_NAME}
!pip install -r requirements.txt

## 5. Train Both Models on GPU
- Trains Baseline Seq2Seq model first
- Trains Bahdanau Attention Seq2Seq model with dynamic context vectors
- Uses CUDA acceleration for fast epochs

In [ ]:
%cd /content/{REPO_NAME}

# Trains both baseline and attention models sequentially
!python3 src/train.py --model all

## 6. Save Trained Checkpoints to Google Drive
Copies `vocab_eng.json`, `vocab_uzb.json`, `seq2seq_attention.pth`, and `seq2seq_baseline.pth` to your Google Drive.

In [ ]:
!cp -rv /content/{REPO_NAME}/checkpoints/* "{DRIVE_DIR}/"
print("\nAll checkpoints & vocabularies successfully backed up to Google Drive!")
!ls -lh "{DRIVE_DIR}"

## 7. Interactive Translation & Attention Matrix Visualizer

In [ ]:
import sys
sys.path.insert(0, f"/content/{REPO_NAME}")
sys.path.insert(0, f"/content/{REPO_NAME}/src")

import torch
from src.dataset import Vocabulary
from src.model_attention import Encoder, Decoder, Seq2SeqAttention
from src.inference import translate_attention, plot_attention_heatmap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eng_vocab = Vocabulary.load("checkpoints/vocab_eng.json")
uzb_vocab = Vocabulary.load("checkpoints/vocab_uzb.json")

state = torch.load("checkpoints/seq2seq_attention.pth", map_location=device)
enc = Encoder(eng_vocab.n_words, state["embedding_dim"], state["hidden_dim"])
dec = Decoder(uzb_vocab.n_words, state["embedding_dim"], state["hidden_dim"])
model = Seq2SeqAttention(enc, dec, device).to(device)
model.load_state_dict(state["model_state_dict"])
model.eval()

test_sentences = [
    "I love reading books in the library.",
    "Tashkent is the capital city of Uzbekistan.",
    "We drank hot green tea with fresh bread."
]

for sent in test_sentences:
    tokens, attn_matrix = translate_attention(sent, model, eng_vocab, uzb_vocab, device=device)
    print("=" * 60)
    print(f"Source:      {sent}")
    print(f"Translation: {' '.join(tokens)}")
    plot_attention_heatmap(sent, tokens, attn_matrix)

## 8. Run Architecture Comparison Benchmark

In [ ]:
%cd /content/{REPO_NAME}
!python3 src/compare.py